 # Day 6 (Thu Aug 20) — Let's build the GPT Tokenizer (2h13m)

He types everything live, so this is a scratchpad, not a starter. His filled colab + minbpe repo are answer keys — stay out of them until it's written.

**why this day matters most:** 28 of the 48 CS336 tests are tokenizer tests (`test_train_bpe.py` 3 + `test_tokenizer.py` 25). It also unblocks the TinyStories run — nothing trains until my tokenizer can encode the corpus.

**arc (video chapters):**
- 14:56 unicode code points · 18:15 UTF-8 and why bytes
- 23:50 BPE algorithm · 27:02 implementation starts
- 28:35 get_stats · 30:36 merge · 34:58 the training loop + compression ratio
- 42:47 decode · 48:21 encode
- 57:36 **regex splitting** — the part that makes it a real tokenizer
- 1:11:38 tiktoken, GPT-2 vs GPT-4 patterns · 1:18:26 special tokens
- 1:25:28 exercise time (build your own GPT-4 tokenizer)
- 1:51:41 the quirks: why models can't spell, 9.11 vs 9.9, SolidGoldMagikarp

**targets:**
1. `train(text, vocab_size)` → vocab + merges. CS336 wants 500-token vocab on `corpus.en` in **under 1.5 seconds** — naive O(n²) passes correctness and fails the clock
2. `encode` / `decode` round-trip on unicode, and matching tiktoken exactly
3. special tokens that never get merged into
4. consolidate into `bpe.py`, wire into `05-cs336/assignment1-basics/tests/adapters.py`

**rule:** watch a chapter → close it → write it here → only then compare to the answer key.

**corpora:** `data/taylorswift.txt` (186KB, karpathy's default for the exercise) · `data/unicode_torture.txt` (fullwidth, circled letters, flag emoji — the round-trip has to survive it). Bigger: `05-cs336/.../tests/fixtures/corpus.en` for the 1.5s speed test.

In [159]:
# hyperparams
VOCAB_SIZE = 276 # the desired final vocabulary size

In [160]:
from pathlib import Path

D = Path("data")
text = (D / "taylorswift.txt").read_text()          # training corpus
torture = (D / "unicode_torture.txt").read_text()   # round-trip must survive this
tokens = text.encode("utf-8")

print(len(text), "chars |", len(tokens), "utf-8 bytes")
print(torture)

185561 chars | 185768 utf-8 bytes
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺🇳🇮🇨🇴🇩🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide.


In [161]:
def get_stats(ids):
  """How often each adjacent pair occurs. {(id, id): count}"""
  counts = {}
  for pair in zip(ids, ids[1:]):
      counts[pair] = counts.get(pair, 0) + 1
  return counts

def merge(ids, pair, idx):
  """Replace every occurrence of `pair` in `ids` with the single token `idx`.
    Non-overlapping, left to right: (32,32) over [32,32,32] -> [idx, 32]."""
  newids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

In [162]:
get_stats(tokens)

{(67, 111): 176,
 (111, 112): 298,
 (112, 121): 6,
 (121, 32): 1248,
 (32, 112): 381,
 (112, 97): 103,
 (97, 115): 559,
 (115, 116): 1089,
 (116, 101): 866,
 (101, 32): 2981,
 (32, 111): 1663,
 (111, 102): 467,
 (102, 32): 464,
 (32, 116): 1824,
 (116, 104): 1737,
 (104, 101): 2118,
 (32, 87): 353,
 (87, 105): 131,
 (105, 107): 44,
 (107, 105): 74,
 (105, 112): 52,
 (112, 101): 267,
 (101, 100): 1876,
 (100, 105): 313,
 (105, 97): 289,
 (97, 32): 420,
 (32, 97): 1495,
 (97, 114): 1519,
 (114, 116): 410,
 (116, 105): 787,
 (105, 99): 678,
 (99, 108): 72,
 (108, 101): 688,
 (111, 110): 1815,
 (110, 32): 1768,
 (32, 84): 934,
 (84, 97): 677,
 (97, 121): 900,
 (121, 108): 707,
 (108, 111): 872,
 (111, 114): 2076,
 (114, 32): 2428,
 (32, 83): 1633,
 (83, 119): 873,
 (119, 105): 1086,
 (105, 102): 955,
 (102, 116): 934,
 (116, 44): 101,
 (44, 32): 2961,
 (115, 32): 2053,
 (32, 70): 356,
 (70, 101): 175,
 (101, 98): 178,
 (98, 32): 17,
 (32, 49): 837,
 (49, 54): 192,
 (54, 44): 150,
 (32, 50)

In [163]:
num_merges = VOCAB_SIZE - 256 # 256 since there are 256 initial vocab size
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (101, 32) into a new token 256
merging (44, 32) into a new token 257
merging (100, 32) into a new token 258
merging (46, 32) into a new token 259
merging (114, 32) into a new token 260
merging (50, 48) into a new token 261
merging (115, 32) into a new token 262
merging (105, 110) into a new token 263
merging (111, 110) into a new token 264
merging (114, 105) into a new token 265
merging (116, 32) into a new token 266
merging (116, 104) into a new token 267
merging (101, 258) into a new token 268
merging (257, 261) into a new token 269
merging (97, 110) into a new token 270
merging (97, 114) into a new token 271
merging (101, 260) into a new token 272
merging (121, 32) into a new token 273
merging (97, 108) into a new token 274
merging (267, 256) into a new token 275


In [164]:
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

[5, 6, 99, 9, 1]


In [165]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 185768
ids length: 147440
compression ratio: 1.26X


In [166]:
vocab = {}
for i in range(256):
    vocab[i] = bytes([i])
for pair, id in merges.items():
    x, y = pair
    vocab[id] = vocab[x] + vocab[y]

vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [167]:
def decode(ids):
    """Given a list of ids, what is the string"""
    tokens = [vocab[i] for i in ids]
    text = b"".join(tokens)
    out = text.decode("utf-8", errors="replace")
    return out

In [168]:
assert decode([128]) == "\ufffd", "lone continuation byte -> replacement char, no crash"
assert decode(list("안녕하세요 👋".encode("utf-8"))) == "안녕하세요 👋", "multi-byte chars must survive"
assert decode([]) == "", "empty in, empty out"
assert decode(ids) == text, "the trained ids must decode back to the whole corpus"
print("decode ok")

decode ok


In [169]:
print(decode(list("안녕하세요 👋".encode("utf-8"))))   # must come back intact
print(decode([128]))                                  # still '�'
print(decode(ids) == text)                            # your trained loop's output

안녕하세요 👋
�
True


In [178]:
merges

{(101, 32): 256,
 (44, 32): 257,
 (100, 32): 258,
 (46, 32): 259,
 (114, 32): 260,
 (50, 48): 261,
 (115, 32): 262,
 (105, 110): 263,
 (111, 110): 264,
 (114, 105): 265,
 (116, 32): 266,
 (116, 104): 267,
 (101, 258): 268,
 (257, 261): 269,
 (97, 110): 270,
 (97, 114): 271,
 (101, 260): 272,
 (121, 32): 273,
 (97, 108): 274,
 (267, 256): 275}

In [186]:
def encode(text):
    """Given a text, what is the id"""
    texts_enc_ids = list(text.encode("utf-8"))
    while len(texts_enc_ids) >= 2:
        stats = get_stats(texts_enc_ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break 
        idx = merges[pair]
        texts_enc_ids = merge(texts_enc_ids, pair, idx)
    return texts_enc_ids

encode("empty in, empty out")

[101, 109, 112, 116, 273, 263, 257, 101, 109, 112, 116, 273, 111, 117, 116]

In [185]:
# encode checks
assert encode("") == [], "empty in, empty out"
assert encode("a") == [97], "single ascii char is its own byte"
assert all(0 <= i < VOCAB_SIZE for i in encode(text)), "no id outside the vocab"

# the one that matters: round-trip
for s in ["", "a", "hello world", "the quick brown fox", torture, "안녕하세요 👋", "   ", "\n\n\t"]:
    assert decode(encode(s)) == s, f"round-trip failed on {s!r}"

# merges must actually be applied — encoding should be shorter than raw bytes
raw = len(text.encode("utf-8"))
enc = len(encode(text))
print(f"raw {raw} -> encoded {enc}  |  compression {raw/enc:.2f}X")
assert enc < raw, "encode is not applying any merges"
assert encode(text) == ids, "encode(text) must reproduce exactly what the training loop produced"
print("encode ok")

raw 185768 -> encoded 147440  |  compression 1.26X
encode ok


In [ ]:
def train(text, vocab_size, verbose=False):
    num_merges = vocab_size - 256
    for i in range(num_merges):
        

In [ ]:
# train checks — compares against the inline loop I already verified
v, m = train(text, VOCAB_SIZE)

assert len(v) == VOCAB_SIZE, f"vocab must be 256 base + merges, got {len(v)}"
assert len(m) == VOCAB_SIZE - 256, f"expected {VOCAB_SIZE-256} merges, got {len(m)}"
assert all(v[i] == bytes([i]) for i in range(256)), "base bytes must be untouched"
assert v == vocab and m == merges, "train must reproduce the inline loop exactly"

# ids are contiguous, and every merged token equals its parents concatenated
assert sorted(m.values()) == list(range(256, VOCAB_SIZE)), "new ids must be 256,257,... in order"
for (a, b), idx in m.items():
    assert v[idx] == v[a] + v[b], f"vocab[{idx}] != vocab[{a}] + vocab[{b}]"

# determinism — no randomness anywhere in BPE
assert train(text, VOCAB_SIZE)[1] == m, "same corpus + same vocab_size must give identical merges"

# smaller vocab is a prefix of a bigger one: merges are learned greedily, in order
v2, m2 = train(text, 266)
assert list(m2.items()) == list(m.items())[:10], "first 10 merges must not depend on how many you ask for"

# a tokenizer trained on different text learns different merges
v3, m3 = train(torture, 276)
assert m3 != m, "different corpus must give a different tokenizer"

print("train ok |", len(v), "vocab,", len(m), "merges")
print("learned:", [v[i] for i in range(256, min(276, VOCAB_SIZE))])

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # GPT-4 tokenizer
print(enc.encode("안녕하세요 👋 (hello in Korean!)"))
print(enc.decode(enc.encode("안녕하세요 👋 (hello in Korean!)")) == "안녕하세요 👋 (hello in Korean!)")

# match the above for your own tokenizer, and also implement a train() function

[31495, 230, 75265, 243, 92245, 62904, 233, 320, 15339, 304, 16526, 16715]
True
